# Ball detector training on Kaggle - far-court focus

Kaggle equivalent of `notebooks/train_ball_detector_farcourt_colab.ipynb`, for when Colab's free GPU quota is exhausted - Kaggle gives free GPU time on a completely separate weekly quota (~30 hrs/week).

Two real differences from the Colab version:

1. **Dataset comes in as a Kaggle Dataset, not a Drive mount.** Before running this notebook: go to kaggle.com/datasets -> New Dataset -> upload `tennis_ball_dataset_colab.zip` (the same 2.5GB zip built locally, already used for the Colab notebooks - `data/raw/train`, `data/raw/valid`, current `weights/ball_detector.pt`, `configs/ball_dataset.yaml`). Kaggle auto-unzips an uploaded .zip into the dataset's file tree. Then in this notebook, right sidebar -> Add Input -> search for the dataset you just created -> add it.
2. **No Drive-equivalent for continuous checkpoint writes.** `/kaggle/working/` is this notebook's persistent output directory - it survives to the end of a run, but if the interactive session disconnects mid-training you lose progress since there's nothing continuously syncing elsewhere the way Colab's `--project` pointed at Drive did. For a run this long, use **Save Version -> Save & Run All (Commit)** (top right) instead of running cells interactively - this runs the whole notebook as a background batch job tied to your account (no browser tab needs to stay open), and `/kaggle/working/` is preserved automatically when it finishes.

**Before running:**
- Settings (right sidebar) -> Accelerator -> GPU T4 x2 or GPU P100.
- Settings -> Internet -> **On** (needed for `git clone` and `pip install` below - off by default).
- Add the dataset as described above, then run the first cell and check the printed path matches what the rest of this notebook expects - Kaggle's mount folder name comes from your dataset's title, not a fixed path, so `DATASET_DIR` below may need editing.

In [ ]:
!ls /kaggle/input/
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# EDIT THIS to match whatever folder the previous cell printed under /kaggle/input/
DATASET_DIR = "/kaggle/input/tennis-ball-dataset-colab"

!find {DATASET_DIR} -maxdepth 3 -type d

In [ ]:
# Clone the repo (public GitHub) at the working branch - needs Settings -> Internet -> On
!git clone -b claude/tennis-ball-yolo-tracking-p8ntwh --depth 1 https://github.com/will-wang1/tennis-tracking-yolo-v1.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

In [ ]:
# Drop the dataset's contents into the repo's expected layout, plus the
# optional far-court hard-positive bundle if you've added a second Kaggle
# Dataset for it (same idea as the Colab notebook's farcourt_hard_positives.zip,
# built via scripts/add_hard_positives.py) - set HARDPOS_DIR if you have one,
# leave as None to train on the base dataset alone (still a real if smaller
# far-court improvement from imgsz=1664 alone).
HARDPOS_DIR = None  # e.g. "/kaggle/input/farcourt-hard-positives"

!rm -rf /kaggle/working/repo/data/raw/train /kaggle/working/repo/data/raw/valid
!mkdir -p /kaggle/working/repo/data/raw
!cp -r {DATASET_DIR}/data/raw/train /kaggle/working/repo/data/raw/train
!cp -r {DATASET_DIR}/data/raw/valid /kaggle/working/repo/data/raw/valid
!cp {DATASET_DIR}/weights/ball_detector.pt /kaggle/working/repo/weights/ball_detector.pt
!cp {DATASET_DIR}/configs/ball_dataset.yaml /kaggle/working/repo/configs/ball_dataset.yaml

if HARDPOS_DIR:
    !cp {HARDPOS_DIR}/images/*.jpg /kaggle/working/repo/data/raw/train/images/
    !cp {HARDPOS_DIR}/labels/*.txt /kaggle/working/repo/data/raw/train/labels/

!ls /kaggle/working/repo/data/raw/train/images | wc -l
!ls /kaggle/working/repo/data/raw/valid/images | wc -l

In [ ]:
# Train at imgsz=1664 for far-court recall, same reasoning as the Colab
# notebook. batch=8 fits a single T4/P100's 16GB comfortably at this
# resolution (same as the Colab version - not using the second GPU on a
# T4 x2 instance, ultralytics needs device="0,1" and a larger batch to
# actually benefit from both, not worth the added complexity here).
# --project writes into /kaggle/working so it's part of this notebook's
# preserved output - see the markdown intro re: Save & Run All (Commit).
!python scripts/train.py \
    --model weights/ball_detector.pt \
    --imgsz 1664 \
    --epochs 40 \
    --batch 8 \
    --patience 15 \
    --name ball_detector_farcourt_kaggle \
    --workers 2 \
    --project /kaggle/working/runs

## After training

Unlike the Colab version, there's no Drive folder to check - once this notebook finishes (or you commit via Save & Run All), the best checkpoint is at `/kaggle/working/runs/ball_detector_farcourt_kaggle/weights/best.pt`, downloadable from this notebook's **Output** tab (or Data tab, depending on Kaggle's current UI) on kaggle.com. On your local machine:

1. Download `best.pt` from the notebook's output.
2. `cp weights/ball_detector.pt weights/ball_detector_pre_farcourt_backup.pt` (back up the current one first)
3. Move the downloaded `best.pt` to `weights/ball_detector.pt`
4. Since this was trained at `imgsz=1664`, pass `--imgsz 1664` to `main.py` too when using it - a mismatched inference resolution gives up most of the far-court benefit this was trained for.
5. `python -m pytest tests/ -q` to sanity check nothing broke.